In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/IQ-OTH_NCCD lung cancer dataset.txt
/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/Normal cases/Normal case (246).jpg
/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/Normal cases/Normal case (155).jpg
/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/Normal cases/Normal case (311).jpg
/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/Normal cases/Normal case (45).jpg
/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/Normal cases/Normal case (298).jpg
/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset/Normal cases/Normal case (359).jpg
/kaggle/input/datasets/hamdallak/the-iqot

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image, ImageFilter
from sklearn.preprocessing import LabelEncoder

# ১. পাথ সেটআপ
base_path = '/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset'
folders = {
    'Bengin cases': 'Benign',
    'Malignant cases': 'Malignant',
    'Normal cases': 'Normal'
}

# ২. ইমেজ পাথ এবং লেবেল সংগ্রহ
data = []
for folder_name, label in folders.items():
    folder_path = os.path.join(base_path, folder_name)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('.jpg', '.png', '.jpeg')):
            data.append((os.path.join(folder_path, img_name), label))

df = pd.DataFrame(data, columns=['image_path', 'label'])

# ৩. লেবেল এনকোডিং (Label Encoding)
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
print(f"Classes found: {le.classes_}")

# ৪. কাস্টম নয়েজ রিডাকশন ফাংশন
class ReduceNoise(object):
    def __call__(self, img):
        return img.filter(ImageFilter.SMOOTH)

# ৫. ডাটা অগমেন্টেশন এবং ট্রান্সফর্মেশন
# আপনার রিকোয়েস্ট অনুযায়ী: Resize(224x224), Rotation(45), Normalization, RGB
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(45), # ৪৫ ডিগ্রি রোটেশন
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # ক্রপিং
    ReduceNoise(), # নয়েজ রিডাকশন
    transforms.ToTensor(), # পিক্সেল ভ্যালু [0, 1] এ নিয়ে আসে
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # VGG16 standard
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    ReduceNoise(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ৬. কাস্টম ডেটাসেট ক্লাস
class LungCancerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]
        image = Image.open(img_path).convert('RGB') # RGB নিশ্চিত করা
        label = self.dataframe.iloc[idx, 2] # Encoded Label
        
        if self.transform:
            image = self.transform(image)
        return image, label

# ৭. ট্রেন, টেস্ট এবং ভ্যালিডেশন স্প্লিট (৭০, ২০, ১০)
train_df = df.sample(frac=0.7, random_state=42)
rem_df = df.drop(train_df.index)
test_df = rem_df.sample(frac=2/3, random_state=42) # অবশিষ্টের ২/৩ মানে মোট ২০%
val_df = rem_df.drop(test_df.index) # অবশিষ্ট ১০%

# ৮. ডেটা লোডার (Batch Size = 3)
train_dataset = LungCancerDataset(train_df, transform=train_transform)
val_dataset = LungCancerDataset(val_df, transform=val_test_transform)
test_dataset = LungCancerDataset(test_df, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=3, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=3)
test_loader = DataLoader(test_dataset, batch_size=3)

# ৯. আউটপুট ফাইল CSV আকারে সেভ করা
df.to_csv('preprocessed_lung_cancer_metadata.csv', index=False)
print("CSV file saved as 'preprocessed_lung_cancer_metadata.csv'")

# চেক করার জন্য ব্যাচ আউটপুট
images, labels = next(iter(train_loader))
print(f"Batch Images Shape: {images.shape}") # [3, 3, 224, 224]

Classes found: ['Benign' 'Malignant' 'Normal']
CSV file saved as 'preprocessed_lung_cancer_metadata.csv'
Batch Images Shape: torch.Size([3, 3, 224, 224])
